# Imports and Configs

In [ ]:
# Install klib and flash libraries
!pip install klib git+https://github.com/flash-lib/flash.git -q

In [ ]:
# Standard Libraries
import toml

# Data Manipulation
import numpy as np
import pandas as pd

# Data Analysis
import klib
import flash as fz
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Change directory to current working directory
%cd /content/drive/MyDrive/Projects/loan-sanction-prediction

In [ ]:
# Load the configurations
with open("config/config.toml", "r") as file:
    config_data = toml.load(file)

num_cols, cat_cols = config_data['num']['cols'], config_data['cat']['cols']

# Preparation

In [ ]:
# Load the dataset
df = pd.read_csv('data/interim/cleaned_train_data_v2.csv')

# Feature construction

- It looks like people with a co-applicant income of 0 doesn't have a co-applicant. So, we should create a new feature called 'has_coapplicant'. For this feature, set the value to 'no' for individuals with a co-applicant income of 0, and 'yes' for those with a non-zero co-applicant income.

In [ ]:
df['has_coapplicant'] = np.where(df['coapplicant_income'] == 0, 'no', 'yes')

# Test
df['has_coapplicant']

In [ ]:
# Appending newly created features based on their feature type
cat_cols.append('has_coapplicant')

# Test
cat_cols

## EDA

### Univariate analysis

In [ ]:
# Statistical measures
df[['has_coapplicant']].describe().T

In [ ]:
# Countplot
sns.countplot(x=df['has_coapplicant'])
plt.show()

### Bivariate analysis

#### Features

##### Categorical - Categorical

In [ ]:
# Heatmap
fz.crosstab_heatmap_viz(df, cat_cols, ['has_coapplicant'], 'both')

##### Numerical - Categorical

In [ ]:
# Point plot
fig, axs = fz.num_cat_viz(df, num_cols, 'has_coapplicant', kind='point')
fig

#### Target

In [ ]:
# Heatmap
fz.crosstab_heatmap_viz(df, ['loan_status'], ['has_coapplicant'], 'both')

# Feature transformation

In [ ]:
transformed_data = fz.feature_transform(df[num_cols])

## applicant_income

In [ ]:
# Histogram
fig, axs = fz.feature_transform_viz(df['applicant_income'], transformed_data)
fig

In [ ]:
# QQ Plot
fig, axs = fz.feature_transform_viz(df['applicant_income'], transformed_data, kind='qq')
fig

## coapplicant_income

In [ ]:
# Histogram
fig, axs = fz.feature_transform_viz(df['coapplicant_income'], transformed_data)
fig

In [ ]:
# QQ Plot
fig, axs = fz.feature_transform_viz(df['coapplicant_income'], transformed_data, kind='qq')
fig

## loan_amount

In [ ]:
# Histogram
fig, axs = fz.feature_transform_viz(df['loan_amount'], transformed_data)
fig

In [ ]:
# QQ Plot
fig, axs = fz.feature_transform_viz(df['loan_amount'], transformed_data, kind='qq')
fig

## Conclusions

- **applicant_income & loan_amount:** Quantile Transform normalizes the data effectively.  
- **coapplicant_income:** Reciprocal Transform transforms coapplicant_income to follow a bimodal distribution.

In [ ]:
df['applicant_income'] = transformed_data['Quantile']['applicant_income']
df['coapplicant_income'] = transformed_data['Reciprocal']['coapplicant_income']
df['loan_amount'] = transformed_data['Quantile']['loan_amount']

In [ ]:
# Test
fig, axs = fz.hist_box_viz(df[num_cols])
fig

# Export

In [ ]:
# Export dataset
fz.export(df, 'data/interim/feature_engineered_train_data_v1.csv')

In [ ]:
# Export metadata as a .toml file to config directory
config_data = {
    "num": {"cols": num_cols},
    "cat": {"cols": cat_cols},
}

with open("config/config_v2.toml", "w") as file:
    toml.dump(config_data, file)

# Test
with open("config/config_v2.toml", "r") as file:
    config_data = toml.load(file)

print(config_data['num']['cols'])